In [2]:
from collections import defaultdict
from csv import DictReader, DictWriter
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path
import re
import unicodedata

DATA_DIR = Path.cwd() / "public" / "data"
if not (DATA_DIR / "ea_fc26_players.csv").exists():
    DATA_DIR = Path.cwd()

EA_PATH = DATA_DIR / "ea_fc26_players.csv"
SQUAD_PATH = DATA_DIR / "SquadLists.csv"
OUTPUT_PATH = DATA_DIR / "playerrating.csv"

NATIONALITY_ALIASES = {
    "the netherlands": "netherlands",
    "holland": "netherlands",
    "nederland": "netherlands",
    "ivory coast": "cote d ivoire",
    "cote divoire": "cote d ivoire",
    "republic of korea": "korea republic",
    "south korea": "korea republic",
    "usa": "united states",
    "u s a": "united states",
    "united states of america": "united states",
}


def clean(value):
    text = unicodedata.normalize("NFKD", str(value or "")).encode("ascii", "ignore").decode("ascii")
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def compact_name(*parts):
    cleaned_parts = []
    for part in parts:
        part_clean = clean(part)
        if part_clean:
            cleaned_parts.append(part_clean)
    return " ".join(cleaned_parts)


def name_variants(*values):
    variants = set()
    for value in values:
        normalized = clean(value)
        if normalized:
            variants.add(normalized)
            variants.add(normalized.replace(" ", ""))
    return variants


def normalize_nationality(value):
    text = clean(value)
    return NATIONALITY_ALIASES.get(text, text)


def parse_squad_dob(value):
    value = str(value or "").strip()
    return datetime.strptime(value, "%d/%m/%Y").date()


def parse_ea_dob(value):
    value = str(value or "").strip()
    return datetime.strptime(value, "%m/%d/%Y %I:%M:%S %p").date()


def squad_name_candidates(row):
    return name_variants(
        row.get("Player Name"),
        compact_name(row.get("First Name(s)"), row.get("Last Name(s)")),
        compact_name(row.get("Last Name(s)"), row.get("First Name(s)")),
        row.get("Name on Shirt"),
    )


def ea_name_candidates(row):
    return name_variants(
        compact_name(row.get("firstName"), row.get("lastName")),
        compact_name(row.get("lastName"), row.get("firstName")),
        row.get("commonName"),
    )


def dedupe(records):
    unique = []
    seen = set()
    for record in records:
        idx = record["_idx"]
        if idx not in seen:
            seen.add(idx)
            unique.append(record)
    return unique


def to_int(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return -1


def score_match(squad_row, ea_row):
    nationality_score = int(squad_row["_nationality"] == ea_row["_nationality"] and squad_row["_nationality"] != "")
    name_overlap = len(squad_row["_names"] & ea_row["_names"])
    rating_score = to_int(ea_row.get("overallRating"))
    return (nationality_score, name_overlap, rating_score)


def name_similarity(squad_row, ea_row):
    best_ratio = 0.0
    for squad_name in squad_row["_names"]:
        for ea_name in ea_row["_names"]:
            best_ratio = max(best_ratio, SequenceMatcher(None, squad_name, ea_name).ratio())
    return best_ratio


def fuzzy_score(squad_row, ea_row):
    similarity = name_similarity(squad_row, ea_row)
    nationality_score = int(squad_row["_nationality"] == ea_row["_nationality"] and squad_row["_nationality"] != "")
    rating_score = to_int(ea_row.get("overallRating"))
    return (similarity, nationality_score, rating_score)


with EA_PATH.open("r", newline="", encoding="utf-8-sig") as handle:
    ea_reader = DictReader(handle)
    ea_rows = []
    for idx, row in enumerate(ea_reader):
        record = dict(row)
        record["_idx"] = idx
        record["_dob"] = parse_ea_dob(row.get("birthdate"))
        record["_nationality"] = normalize_nationality(row.get("nationality"))
        record["_names"] = ea_name_candidates(row)
        ea_rows.append(record)

name_dob_index = defaultdict(list)
name_dob_nationality_index = defaultdict(list)
dob_index = defaultdict(list)
for row in ea_rows:
    if row["_dob"] is not None:
        dob_index[row["_dob"]].append(row)
        for name in row["_names"]:
            name_dob_index[(name, row["_dob"])] .append(row)
            name_dob_nationality_index[(name, row["_dob"], row["_nationality"])] .append(row)

with SQUAD_PATH.open("r", newline="", encoding="utf-8-sig") as handle:
    squad_reader = DictReader(handle)
    squad_headers = squad_reader.fieldnames or []
    squad_rows = []
    for idx, row in enumerate(squad_reader):
        record = dict(row)
        record["_idx"] = idx
        record["_dob"] = parse_squad_dob(row.get("DOB"))
        record["_nationality"] = normalize_nationality(row.get("Team"))
        record["_names"] = squad_name_candidates(row)
        squad_rows.append(record)


def best_from_candidates(candidates, score_fn):
    if not candidates:
        return None
    unique_candidates = dedupe(candidates)
    return max(unique_candidates, key=score_fn)


def find_best_match(squad_row):
    if squad_row["_dob"] is None:
        return None

    exact_candidates = []
    for name in squad_row["_names"]:
        exact_candidates.extend(name_dob_nationality_index.get((name, squad_row["_dob"], squad_row["_nationality"]), []))
    match = best_from_candidates(exact_candidates, lambda ea_row: score_match(squad_row, ea_row))
    if match is not None:
        return match

    fallback_candidates = []
    for name in squad_row["_names"]:
        fallback_candidates.extend(name_dob_index.get((name, squad_row["_dob"]), []))
    match = best_from_candidates(fallback_candidates, lambda ea_row: score_match(squad_row, ea_row))
    if match is not None:
        return match

    fuzzy_candidates = dob_index.get(squad_row["_dob"], [])
    best_match = None
    best_score = None
    for ea_row in fuzzy_candidates:
        current_score = fuzzy_score(squad_row, ea_row)
        if best_score is None or current_score > best_score:
            best_score = current_score
            best_match = ea_row
    if best_score is not None and best_score[0] >= 0.92:
        return best_match

    return None


for row in squad_rows:
    row["_match"] = find_best_match(row)

output_headers = list(squad_headers) + ["overallRating"]
with OUTPUT_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = DictWriter(handle, fieldnames=output_headers)
    writer.writeheader()
    for row in squad_rows:
        output_row = {field: row.get(field, "") for field in squad_headers}
        output_row["overallRating"] = row["_match"].get("overallRating", "") if row["_match"] else ""
        writer.writerow(output_row)

matched_count = sum(1 for row in squad_rows if row["_match"])
unmatched_rows = [row for row in squad_rows if not row["_match"]]
print(f"Matched {matched_count} of {len(squad_rows)} squad rows.")
print(f"Wrote {OUTPUT_PATH}")
if unmatched_rows:
    print(f"Unmatched rows: {len(unmatched_rows)}")
    print("First few unmatched players:")
    for row in unmatched_rows[:10]:
        print(f"- {row.get('Player Name', '')} | {row.get('Team', '')} | {row.get('DOB', '')}")

Matched 859 of 1248 squad rows.
Wrote d:\DeskT\WebD\Games\fbdraft\public\data\playerrating.csv
Unmatched rows: 389
First few unmatched players:
- MASTIL Melvin | Algeria | 19/02/2000
- ABADA Achref | Algeria | 15/06/1999
- TOUGAI Mohamed Amine | Algeria | 22/01/2000
- BELAID Zineddine | Algeria | 20/03/1999
- BENBOUALI Nadhir | Algeria | 17/04/2000
- BENBOT Oussama | Algeria | 11/10/1994
- BOULBINA Adil | Algeria | 02/05/2003
- TITRAOUI Yassine | Algeria | 26/07/2003
- LOPEZ Jose Manuel | Argentina | 06/12/2000
- RYAN Mathew | Australia | 08/04/1992


In [5]:
from csv import DictReader
from json import dump
from pathlib import Path
import re
import unicodedata

DATA_DIR = Path.cwd() / "public" / "data"
if not (DATA_DIR / "ea_fc26_players.csv").exists():
    DATA_DIR = Path.cwd()

EA_PATH = DATA_DIR / "ea_fc26_players.csv"
TOP500_PATH = DATA_DIR / "top500.json"

def clean(value):
    text = unicodedata.normalize("NFKD", str(value or "")).encode("ascii", "ignore").decode("ascii")
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_rating(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return -1


def parse_id(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return 10**12


def split_list_field(value):
    return [item.strip() for item in str(value or "").split(",") if item.strip()]


def player_name(row):
    common_name = str(row.get("commonName") or "").strip()
    if common_name:
        return common_name
    first_name = str(row.get("firstName") or "").strip()
    last_name = str(row.get("lastName") or "").strip()
    combined_name = f"{first_name} {last_name}".strip()
    return combined_name or str(row.get("id") or "").strip()


def position_bucket(row):
    position_type = clean(row.get("positionType"))
    position = clean(row.get("position"))
    if position_type in {"goalkeeper", "gk"} or position == "gk":
        return "Goalkeepers"
    if position_type in {"defense", "defence", "defender"} or position in {"cb", "lb", "rb", "lwb", "rwb"}:
        return "Defenders"
    if position_type in {"midfielder", "midfield"} or position in {"cdm", "cm", "cam", "lm", "rm"}:
        return "Midfielders"
    if position_type in {"attack", "attacker", "forward"} or position in {"st", "cf", "lw", "rw"}:
        return "Attackers"
    return "Remaining"


quotas = {
    "Goalkeepers": 40,
    "Defenders": 100,
    "Midfielders": 100,
    "Attackers": 100,
}

with EA_PATH.open("r", newline="", encoding="utf-8-sig") as handle:
    reader = DictReader(handle)
    players = []
    for row in reader:
        rating = parse_rating(row.get("overallRating"))
        if rating < 0:
            continue
        positions = [row.get("position")] + split_list_field(row.get("alternatePositions"))
        positions = [position.strip() for position in positions if str(position or "").strip()]
        traits = split_list_field(row.get("playStyles"))
        players.append({
            "id": parse_id(row.get("id")),
            "player name": player_name(row),
            "country": str(row.get("nationality") or "").strip(),
            "rating": rating,
            "traits": traits,
            "positions": positions,
            "_bucket": position_bucket(row),
            "_sort_key": (-rating, parse_id(row.get("id"))),
        })

selected = []
selected_ids = set()

for bucket, quota in quotas.items():
    bucket_players = sorted(
        [player for player in players if player["_bucket"] == bucket],
        key=lambda player: player["_sort_key"],
    )
    for player in bucket_players[:quota]:
        if player["id"] not in selected_ids:
            selected.append(player)
            selected_ids.add(player["id"])

remaining_players = sorted(
    [player for player in players if player["id"] not in selected_ids],
    key=lambda player: player["_sort_key"],
 )
for player in remaining_players:
    if len(selected) >= 500:
        break
    selected.append(player)
    selected_ids.add(player["id"])

selected = sorted(selected, key=lambda player: player["_sort_key"])[:500]

output = [
    {
        "id": player["id"],
        "player name": player["player name"],
        "country": player["country"],
        "rating": player["rating"],
        "traits": player["traits"],
        "positions": player["positions"],
    }
    for player in selected
 ]

with TOP500_PATH.open("w", encoding="utf-8") as handle:
    dump(output, handle, ensure_ascii=False, indent=2)
    handle.write("\n")

bucket_counts = {bucket: sum(1 for player in selected if player["_bucket"] == bucket) for bucket in quotas}
bucket_counts["Remaining"] = sum(1 for player in selected if player["_bucket"] == "Remaining")
print(f"Wrote {TOP500_PATH}")
print(f"Total rows: {len(selected)}")
for bucket in ["Goalkeepers", "Defenders", "Midfielders", "Attackers", "Remaining"]:
    print(f"{bucket}: {bucket_counts[bucket]}")

Wrote d:\DeskT\WebD\Games\fbdraft\public\data\top500.json
Total rows: 500
Goalkeepers: 57
Defenders: 139
Midfielders: 199
Attackers: 105
Remaining: 0


In [ ]:
from csv import DictReader
from json import dump
from pathlib import Path
import re
import unicodedata

DATA_DIR = Path.cwd() / "public" / "data"
if not (DATA_DIR / "ea_fc26_players.csv").exists():
    DATA_DIR = Path.cwd()

EA_PATH = DATA_DIR / "ea_fc26_players.csv"
TOP500_PATH = DATA_DIR / "top500.json"


def clean(value):
    text = unicodedata.normalize("NFKD", str(value or "")).encode("ascii", "ignore").decode("ascii")
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_rating(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return -1


def parse_id(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return 10**12


def split_list_field(value):
    return [item.strip() for item in str(value or "").split(",") if item.strip()]


def player_name(row):
    common_name = str(row.get("commonName") or "").strip()
    if common_name:
        return common_name
    first_name = str(row.get("firstName") or "").strip()
    last_name = str(row.get("lastName") or "").strip()
    combined_name = f"{first_name} {last_name}".strip()
    return combined_name or str(row.get("id") or "").strip()


def normalized_positions(row):
    positions = [str(row.get("position") or "").strip()]
    positions.extend(split_list_field(row.get("alternatePositions")))
    seen = set()
    result = []
    for position in positions:
        if position and position not in seen:
            seen.add(position)
            result.append(position)
    return result


def position_bucket(row):
    position_type = clean(row.get("positionType"))
    position = clean(row.get("position"))
    if position_type in {"goalkeeper", "gk"} or position == "gk":
        return "Goalkeepers"
    if position_type in {"defense", "defence", "defender"} or position in {"cb", "lb", "rb", "lwb", "rwb"}:
        return "Defenders"
    if position_type in {"midfielder", "midfield"} or position in {"cdm", "cm", "cam", "lm", "rm"}:
        return "Midfielders"
    if position_type in {"attack", "attacker", "forward"} or position in {"st", "cf", "lw", "rw"}:
        return "Attackers"
    return "Remaining"


quotas = {
    "Goalkeepers": 40,
    "Defenders": 100,
    "Midfielders": 100,
    "Attackers": 100,
}

with EA_PATH.open("r", newline="", encoding="utf-8-sig") as handle:
    reader = DictReader(handle)
    players = []
    for row in reader:
        rating = parse_rating(row.get("overallRating"))
        if rating < 0:
            continue
        player_id = parse_id(row.get("id"))
        players.append({
            "id": player_id,
            "player name": player_name(row),
            "country": str(row.get("nationality") or "").strip(),
            "rating": rating,
            "traits": split_list_field(row.get("playStyles")),
            "positions": normalized_positions(row),
            "_bucket": position_bucket(row),
            "_sort_key": (-rating, player_id),
        })

selected = []
selected_ids = set()

for bucket, quota in quotas.items():
    bucket_players = sorted(
        [player for player in players if player["_bucket"] == bucket],
        key=lambda player: player["_sort_key"],
    )
    for player in bucket_players[:quota]:
        if player["id"] not in selected_ids:
            selected.append(player)
            selected_ids.add(player["id"])

remaining_players = sorted(
    [player for player in players if player["id"] not in selected_ids],
    key=lambda player: player["_sort_key"],
)
for player in remaining_players:
    if len(selected) >= 500:
        break
    selected.append(player)
    selected_ids.add(player["id"])

selected = sorted(selected, key=lambda player: player["_sort_key"])[:500]
output = [
    {
        "id": player["id"],
        "player name": player["player name"],
        "country": player["country"],
        "rating": player["rating"],
        "traits": player["traits"],
        "positions": player["positions"],
    }
    for player in selected
]

with TOP500_PATH.open("w", encoding="utf-8") as handle:
    dump(output, handle, ensure_ascii=False, indent=2)
    handle.write("\n")

bucket_counts = {bucket: sum(1 for player in selected if player["_bucket"] == bucket) for bucket in quotas}
bucket_counts["Remaining"] = sum(1 for player in selected if player["_bucket"] == "Remaining")
print(f"Wrote {TOP500_PATH}")
print(f"Total rows: {len(selected)}")
for bucket in ["Goalkeepers", "Defenders", "Midfielders", "Attackers", "Remaining"]:
    print(f"{bucket}: {bucket_counts[bucket]}")